# Intervention Effectiveness Analyzer
## End-to-End ML Pipeline · BrightHut / Lighthouse Sanctuary

---

## 1. Problem Framing

### Business Problem

BrightHut delivers a range of structured interventions to residents: individual and group counseling sessions, educational support programs, health and medical services, family contact and home visitation, vocational training, and crisis response. With limited staff managing multiple safehouses and dozens of active cases, the organization cannot deliver every intervention to every resident at maximum intensity. They must choose — and their choices determine outcomes.

Currently those choices are driven by professional judgment and protocol, but not by systematic evidence from the organization's own data. The founders want to understand: **Which interventions are actually producing measurable improvements in the girls they serve?** Is individual counseling more effective than group counseling for emotional recovery? Does educational engagement accelerate overall wellbeing? Are health-focused interventions associated with reductions in behavioral incidents? What combination of interventions is most consistently associated with progress?

**The business question is:** *Which intervention types, frequencies, durations, and combinations are most strongly associated with measurable improvements in resident emotional state, health outcomes, and educational engagement — and can we predict which residents are most likely to improve under a given intervention plan?*

### Who Cares About This

| Stakeholder | How they use this model |
|---|---|
| Social Workers | Design evidence-informed intervention plans at case intake |
| Program Directors | Allocate limited staff hours to the interventions with the strongest evidence base |
| Case Conference Coordinators | Adjust plans mid-course when progress stalls |
| Executive Director / Board | Report to donors that program decisions are evidence-based |
| Donor-Facing Dashboard | Demonstrate that donations are funding interventions with measurable outcomes |

### Prediction vs. Explanation — Explicit Choice

This pipeline is primarily **explanatory**, with a secondary predictive component:

- **Explanatory goal (primary):** Use regularized OLS and logistic regression to quantify which intervention types and design attributes are most associated with composite resident improvement. The organization needs to understand *why* certain plans work so it can design better ones going forward. Interpretable coefficients with confidence intervals are the primary output.

- **Predictive goal (secondary):** Train a gradient-boosted classifier to predict whether a resident will show measurable improvement under their current plan. This surfaces residents whose trajectories are predicted to stall — a triage signal for plan revision.

As the textbook emphasizes: **for explanatory work, a model that sacrifices some predictive accuracy to produce clear, defensible estimates is preferable to a black-box model with marginally better predictions.** We explicitly do not use the ensemble model for causal inference.

### Defining "Improvement"

A resident shows **measurable improvement** if they demonstrate positive change across the domains tracked in the dataset. We construct a **composite improvement score** from three behavioral domains:

1. **Emotional domain:** The slope of the resident's emotion scale scores across process recordings is positive (they are trending toward higher emotional wellbeing over their recorded sessions).
2. **Health domain:** The resident's most recent health/wellbeing score is above their intake baseline score.
3. **Education domain:** The resident's attendance or academic performance in their most recent education record is above their earliest recorded baseline.

**Binary label:** `improved = 1` if the resident shows positive direction in **at least 2 of the 3 domains**. This majority-of-domains approach prevents a single strong signal in one domain from masking stagnation in others.

Residents with fewer than 2 process recordings are excluded from training (insufficient behavioral history). They appear in the inference output flagged as "insufficient data."

### Success Metrics

| Metric | Rationale |
|---|---|
| OLS adjusted R² | Explanatory power of intervention attributes on continuous improvement score |
| Coefficient p-values | Statistical defensibility of effect size claims |
| ROC-AUC (GBT) | Out-of-sample discriminative power on held-out residents |
| Recall @ threshold | Missing a resident predicted to stall but who would have benefited from plan revision is a care failure |

**Error cost asymmetry:** A false positive (predicting improvement that doesn't materialize) means a resident stays on a plan that isn't working — a missed opportunity to adjust early. A false negative (predicting stall in a resident who actually improves) is a benign error — the plan works and staff can deprioritize review. **We favor recall in the predictive model**, consistent with the child welfare context.

### Important Limitations

Causal identification in intervention research is deeply challenging. Residents are not randomly assigned to interventions — social workers assign more intensive interventions to residents with more severe presentations. This **selection bias** means that a high-intensity intervention being negatively correlated with improvement in raw data does not imply the intervention is harmful; it may mean it is assigned to the most difficult cases. The OLS coefficients should be interpreted as *associations controlling for observed baseline severity*, not as causal treatment effects. With a small sample of approximately 60 residents, all findings should be treated as directional hypotheses to validate prospectively.

---
## 2. Data Acquisition, Preparation & Exploration

In [ ]:
# ── Install / import ─────────────────────────────────────────────────────────
# Uncomment the line below if running in a fresh Colab / minimal environment
# !pip install scikit-learn pandas numpy matplotlib seaborn requests joblib statsmodels

import warnings, json
warnings.filterwarnings('ignore')

import requests
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import statsmodels.api as sm
import joblib
from pathlib  import Path

from sklearn.pipeline        import Pipeline
from sklearn.preprocessing   import StandardScaler
from sklearn.linear_model    import LogisticRegression, Ridge
from sklearn.ensemble        import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score, train_test_split
from sklearn.metrics         import (
    roc_auc_score, classification_report,
    ConfusionMatrixDisplay, fbeta_score, precision_recall_curve
)
from sklearn.inspection      import permutation_importance

RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)
plt.rcParams['figure.dpi'] = 110
sns.set_theme(style='whitegrid', palette='muted')
print('Imports OK')

In [ ]:
# ── Emotion scale encoding ────────────────────────────────────────────────────
# Ordinal mapping from text descriptors to numeric 1–8 scale.
# Used to convert free-text emotional state fields in process_recordings.

EMOTION_SCALE = {
    'distressed': 1, 'crisis': 1, 'severe': 1,
    'anxious': 2, 'fearful': 2, 'withdrawn': 2, 'depressed': 2,
    'sad': 3, 'frustrated': 3, 'angry': 3, 'irritable': 3,
    'neutral': 4,
    'calm': 5, 'stable': 5,
    'hopeful': 6, 'content': 6,
    'happy': 7, 'engaged': 7, 'positive': 7,
    'thriving': 8, 'excellent': 8,
}

def encode_emotion(text_val) -> float:
    """Map a text emotion descriptor to a numeric score."""
    if pd.isna(text_val):
        return np.nan
    text = str(text_val).lower().strip()
    # Try direct match first, then substring match
    if text in EMOTION_SCALE:
        return float(EMOTION_SCALE[text])
    for key, val in EMOTION_SCALE.items():
        if key in text:
            return float(val)
    # Try numeric parse as fallback
    try:
        return float(text)
    except ValueError:
        return np.nan

def compute_slope(series: pd.Series) -> float:
    """Linear trend slope over time-ordered values. Returns 0 if fewer than 2 points."""
    s = series.dropna()
    if len(s) < 2:
        return 0.0
    x = np.arange(len(s))
    return float(np.polyfit(x, s.values, 1)[0])

print('Emotion scale and helpers defined.')

In [ ]:
# ── Load tables from the BrightHut REST API ───────────────────────────────────
# All data is sourced from the live deployed backend — no local CSV files.
# Endpoint pattern: GET /api/tables/{tableName}  (no auth required for reads)

API_BASE = 'https://brighthut-befxhqfdabcpfscu.centralus-01.azurewebsites.net'

def load_table(name: str) -> pd.DataFrame:
    """Fetch an entire table from the BrightHut REST API."""
    url = f'{API_BASE}/api/tables/{name}'
    resp = requests.get(url, timeout=30)
    resp.raise_for_status()
    df = pd.DataFrame(resp.json())
    print(f'  {name:40s}  {df.shape[0]:>5} rows  {df.shape[1]:>3} cols')
    return df

def find_col(df: pd.DataFrame, *keywords) -> str | None:
    """Return the first column name containing any of the keywords (case-insensitive)."""
    for kw in keywords:
        for col in df.columns:
            if kw.lower() in col.lower():
                return col
    return None

print('Loading tables...')
residents     = load_table('residents')
interventions = load_table('intervention_plans')
recordings    = load_table('process_recordings')
health_recs   = load_table('health_wellbeing_records')
edu_recs      = load_table('education_records')
incidents     = load_table('incident_reports')
visitations   = load_table('home_visitations')
print('Done.')

In [ ]:
# ── Schema inspection ─────────────────────────────────────────────────────────
print('=== intervention_plans columns ===')
print(interventions.dtypes)
print('\n=== process_recordings columns ===')
print(recordings.dtypes)
print('\n=== health_wellbeing_records columns ===')
print(health_recs.dtypes)
print('\n=== education_records columns ===')
print(edu_recs.dtypes)
print('\nSample interventions row:')
interventions.head(2)

In [ ]:
# ── Resolve dynamic column names ─────────────────────────────────────────────

# residents
col_res_id       = find_col(residents, 'resident_id', 'id')
col_res_dob      = find_col(residents, 'date_of_birth', 'dob', 'birthdate', 'birth_date')
col_res_admit    = find_col(residents, 'admission_date', 'admit_date', 'intake_date', 'date')
col_res_category = find_col(residents, 'case_category', 'category', 'case_type', 'type')
col_res_status   = find_col(residents, 'reintegration_status', 'status', 'case_status')

# intervention_plans
col_int_id       = find_col(interventions, 'plan_id', 'intervention_id', 'id')
col_int_res_id   = find_col(interventions, 'resident_id', 'resident')
col_int_type     = find_col(interventions, 'intervention_type', 'type', 'plan_type')
col_int_start    = find_col(interventions, 'start_date', 'plan_date', 'date', 'created')
col_int_end      = find_col(interventions, 'end_date', 'target_date', 'completion_date')
col_int_freq     = find_col(interventions, 'frequency', 'sessions_per_week', 'sessions_per_month')
col_int_status   = find_col(interventions, 'status', 'plan_status', 'completion_status')
col_int_goal     = find_col(interventions, 'goal', 'objective', 'target', 'description')

# process_recordings
col_rec_res_id   = find_col(recordings, 'resident_id', 'resident')
col_rec_date     = find_col(recordings, 'session_date', 'recording_date', 'date')
col_rec_emotion  = find_col(recordings, 'emotional_state', 'emotion', 'emotional', 'mood', 'affect')
col_rec_type     = find_col(recordings, 'session_type', 'type')

# health_wellbeing_records
col_hlth_res_id  = find_col(health_recs, 'resident_id', 'resident')
col_hlth_date    = find_col(health_recs, 'record_date', 'date', 'assessment_date')
col_hlth_score   = find_col(health_recs, 'wellbeing_score', 'health_score', 'score',
                             'overall_score', 'rating')

# education_records
col_edu_res_id   = find_col(edu_recs, 'resident_id', 'resident')
col_edu_date     = find_col(edu_recs, 'record_date', 'date', 'term_start', 'period')
col_edu_attend   = find_col(edu_recs, 'attendance', 'attendance_rate', 'days_attended')
col_edu_perf     = find_col(edu_recs, 'performance', 'grade', 'gpa', 'academic_score', 'score')

# incidents
col_inc_res_id   = find_col(incidents, 'resident_id', 'resident')
col_inc_date     = find_col(incidents, 'incident_date', 'date', 'report_date')
col_inc_severity = find_col(incidents, 'severity', 'level', 'category', 'type')

print('Column resolution complete.')
key_cols = [
    ('resident_id (res)',    col_res_id),
    ('intervention_type',    col_int_type),
    ('int_start',            col_int_start),
    ('int_freq',             col_int_freq),
    ('rec_resident_id',      col_rec_res_id),
    ('rec_emotion',          col_rec_emotion),
    ('rec_date',             col_rec_date),
    ('hlth_score',           col_hlth_score),
    ('edu_attendance',       col_edu_attend),
    ('edu_performance',      col_edu_perf),
]
for alias, actual in key_cols:
    print(f'  {alias:26s} → {actual}')

In [ ]:
# ── Parse dates and numeric columns ──────────────────────────────────────────
REF_DATE = pd.Timestamp.now().normalize()

# Residents
if col_res_admit:
    residents['admit_date'] = pd.to_datetime(residents[col_res_admit], errors='coerce')
else:
    residents['admit_date'] = pd.NaT
if col_res_dob:
    residents['dob'] = pd.to_datetime(residents[col_res_dob], errors='coerce')
    residents['age_at_admit'] = (
        (residents['admit_date'] - residents['dob']).dt.days / 365.25
    ).clip(lower=0, upper=30)
else:
    residents['age_at_admit'] = np.nan

# Intervention plans
if col_int_start:
    interventions['int_start'] = pd.to_datetime(interventions[col_int_start], errors='coerce')
else:
    interventions['int_start'] = pd.NaT
if col_int_end:
    interventions['int_end'] = pd.to_datetime(interventions[col_int_end], errors='coerce')
else:
    interventions['int_end'] = pd.NaT
interventions['int_duration_days'] = (
    interventions['int_end'].fillna(REF_DATE) - interventions['int_start']
).dt.days.clip(lower=0)

# Process recordings
if col_rec_date:
    recordings['rec_date'] = pd.to_datetime(recordings[col_rec_date], errors='coerce')
else:
    recordings['rec_date'] = pd.NaT
if col_rec_emotion:
    recordings['emotion_score'] = recordings[col_rec_emotion].apply(encode_emotion)
else:
    recordings['emotion_score'] = np.nan

# Health records
if col_hlth_date:
    health_recs['hlth_date'] = pd.to_datetime(health_recs[col_hlth_date], errors='coerce')
if col_hlth_score:
    health_recs['hlth_score'] = pd.to_numeric(health_recs[col_hlth_score], errors='coerce')

# Education records
if col_edu_date:
    edu_recs['edu_date'] = pd.to_datetime(edu_recs[col_edu_date], errors='coerce')
if col_edu_attend:
    edu_recs['attendance'] = pd.to_numeric(edu_recs[col_edu_attend], errors='coerce')
if col_edu_perf:
    edu_recs['performance'] = pd.to_numeric(edu_recs[col_edu_perf], errors='coerce')

# Incidents
if col_inc_date:
    incidents['inc_date'] = pd.to_datetime(incidents[col_inc_date], errors='coerce')

print('Date parsing complete.')
print(f'Residents:      {len(residents)}')
print(f'Interventions:  {len(interventions)}')
print(f'Recordings:     {len(recordings)}')
print(f'Health records: {len(health_recs)}')
print(f'Edu records:    {len(edu_recs)}')

In [ ]:
# ── Build per-resident outcome signals ───────────────────────────────────────
# For each resident, compute improvement direction in each of the three domains.

resident_ids = residents[col_res_id].dropna().unique() if col_res_id else []
outcome_rows = []

for rid in resident_ids:
    row = {col_res_id: rid}

    # ── Emotional domain ──────────────────────────────────────────────────────
    rec_sub = recordings[
        recordings[col_rec_res_id] == rid
    ].sort_values('rec_date') if col_rec_res_id else pd.DataFrame()

    emotion_scores = rec_sub['emotion_score'].dropna() if 'emotion_score' in rec_sub else pd.Series()
    row['n_recordings']         = len(rec_sub)
    row['emotion_slope']        = compute_slope(emotion_scores)
    row['avg_emotion']          = float(emotion_scores.mean()) if len(emotion_scores) > 0 else np.nan
    row['first_emotion']        = float(emotion_scores.iloc[0])  if len(emotion_scores) > 0 else np.nan
    row['last_emotion']         = float(emotion_scores.iloc[-1]) if len(emotion_scores) > 0 else np.nan
    row['emotion_improved']     = int(row['emotion_slope'] > 0) if not np.isnan(row['emotion_slope']) else 0

    # Individual vs. group session counts
    if col_rec_type and col_rec_type in rec_sub.columns:
        sess_types = rec_sub[col_rec_type].astype(str).str.lower()
        row['n_individual_sessions'] = int(sess_types.str.contains('individual').sum())
        row['n_group_sessions']      = int(sess_types.str.contains('group').sum())
    else:
        row['n_individual_sessions'] = 0
        row['n_group_sessions']      = 0

    # ── Health domain ─────────────────────────────────────────────────────────
    hlth_sub = health_recs[
        health_recs[col_hlth_res_id] == rid
    ].sort_values('hlth_date') if col_hlth_res_id and 'hlth_date' in health_recs else pd.DataFrame()

    hlth_scores = hlth_sub['hlth_score'].dropna() if 'hlth_score' in hlth_sub else pd.Series()
    row['n_health_records']     = len(hlth_sub)
    row['first_health_score']   = float(hlth_scores.iloc[0])  if len(hlth_scores) > 0 else np.nan
    row['last_health_score']    = float(hlth_scores.iloc[-1]) if len(hlth_scores) > 0 else np.nan
    row['health_slope']         = compute_slope(hlth_scores)
    row['health_improved']      = int(
        row['last_health_score'] > row['first_health_score']
    ) if not (np.isnan(row['first_health_score']) or np.isnan(row['last_health_score'])) else 0

    # ── Education domain ──────────────────────────────────────────────────────
    edu_sub = edu_recs[
        edu_recs[col_edu_res_id] == rid
    ].sort_values('edu_date') if col_edu_res_id and 'edu_date' in edu_recs else pd.DataFrame()

    attend = edu_sub['attendance'].dropna() if 'attendance' in edu_sub else pd.Series()
    perf   = edu_sub['performance'].dropna() if 'performance' in edu_sub else pd.Series()

    row['n_edu_records']        = len(edu_sub)
    row['edu_attend_slope']     = compute_slope(attend)
    row['edu_perf_slope']       = compute_slope(perf)
    row['edu_enrolled']         = int(len(edu_sub) > 0)
    row['education_improved']   = int(
        row['edu_attend_slope'] > 0 or row['edu_perf_slope'] > 0
    )

    # ── Incident domain ───────────────────────────────────────────────────────
    inc_sub = incidents[
        incidents[col_inc_res_id] == rid
    ] if col_inc_res_id else pd.DataFrame()
    row['n_incidents_total']    = len(inc_sub)

    # Recent vs. early incident rate
    if col_res_admit and 'admit_date' in residents.columns:
        admit_dt = residents.loc[residents[col_res_id] == rid, 'admit_date']
        admit_dt = admit_dt.iloc[0] if len(admit_dt) > 0 else pd.NaT
        if not pd.isna(admit_dt) and len(inc_sub) > 0 and 'inc_date' in inc_sub.columns:
            stay_days   = max((REF_DATE - admit_dt).days, 1)
            mid_point   = admit_dt + pd.Timedelta(days=stay_days // 2)
            early_inc   = len(inc_sub[inc_sub['inc_date'] < mid_point])
            recent_inc  = len(inc_sub[inc_sub['inc_date'] >= mid_point])
            row['incident_reduction'] = int(recent_inc < early_inc)
        else:
            row['incident_reduction'] = 0
    else:
        row['incident_reduction'] = 0

    # ── Composite improvement label ────────────────────────────────────────────
    domains_improved = (
        row['emotion_improved'] + row['health_improved'] + row['education_improved']
    )
    row['domains_improved']  = domains_improved
    row['improved']          = int(domains_improved >= 2)

    outcome_rows.append(row)

outcomes = pd.DataFrame(outcome_rows)
print(f'Outcome signals built for {len(outcomes)} residents.')
print(f'Improvement rate (>= 2 domains): {outcomes["improved"].mean():.2%}')
print(f'Domain breakdown:')
print(f'  Emotional improved:  {outcomes["emotion_improved"].mean():.2%}')
print(f'  Health improved:     {outcomes["health_improved"].mean():.2%}')
print(f'  Education improved:  {outcomes["education_improved"].mean():.2%}')

In [ ]:
# ── Build per-resident intervention features ──────────────────────────────────
# Aggregate intervention_plans to one row per resident.

int_feature_rows = []

for rid in resident_ids:
    row = {col_res_id: rid}

    int_sub = interventions[
        interventions[col_int_res_id] == rid
    ] if col_int_res_id else pd.DataFrame()

    row['n_interventions']       = len(int_sub)

    if len(int_sub) == 0:
        row.update({
            'n_intervention_types':    0,
            'total_int_duration_days': 0,
            'avg_int_duration_days':   0,
            'has_counseling':          0,
            'has_group_therapy':       0,
            'has_medical':             0,
            'has_educational_support': 0,
            'has_vocational':          0,
            'has_family_counseling':   0,
            'has_crisis_intervention': 0,
            'pct_completed_plans':     0,
            'avg_freq':                0,
            'intervention_breadth':    0,
        })
        int_feature_rows.append(row)
        continue

    # Duration
    row['total_int_duration_days'] = float(int_sub['int_duration_days'].sum())
    row['avg_int_duration_days']   = float(int_sub['int_duration_days'].mean())

    # Completion rate
    if col_int_status and col_int_status in int_sub.columns:
        completed = int_sub[col_int_status].astype(str).str.lower().str.contains(
            'complete|done|finished|closed', na=False
        )
        row['pct_completed_plans'] = float(completed.mean())
    else:
        row['pct_completed_plans'] = 0.0

    # Frequency (sessions per week / month from column or estimate)
    if col_int_freq and col_int_freq in int_sub.columns:
        freq_num = pd.to_numeric(int_sub[col_int_freq], errors='coerce')
        row['avg_freq'] = float(freq_num.mean()) if freq_num.notna().any() else 0.0
    else:
        row['avg_freq'] = 0.0

    # Intervention type flags (from text matching on type/goal columns)
    type_text = ''
    for tc in [col_int_type, col_int_goal]:
        if tc and tc in int_sub.columns:
            type_text += ' ' + int_sub[tc].astype(str).str.lower().str.cat(sep=' ')

    row['has_counseling']          = int('individual counseling' in type_text or
                                         'individual therapy'   in type_text or
                                         'psychosocial'         in type_text)
    row['has_group_therapy']       = int('group'     in type_text)
    row['has_medical']             = int('medical'   in type_text or
                                         'health'    in type_text or
                                         'clinical'  in type_text)
    row['has_educational_support'] = int('educat'    in type_text or
                                         'school'    in type_text or
                                         'tutoring'  in type_text or
                                         'academic'  in type_text)
    row['has_vocational']          = int('vocational' in type_text or
                                         'livelihood' in type_text or
                                         'skills'     in type_text or
                                         'training'   in type_text)
    row['has_family_counseling']   = int('family'   in type_text or
                                         'parenting' in type_text or
                                         'home visit' in type_text)
    row['has_crisis_intervention'] = int('crisis'   in type_text or
                                         'emergency' in type_text)

    # Breadth: number of distinct intervention types present
    flag_cols = [
        'has_counseling', 'has_group_therapy', 'has_medical',
        'has_educational_support', 'has_vocational',
        'has_family_counseling', 'has_crisis_intervention'
    ]
    row['intervention_breadth']  = sum(row[c] for c in flag_cols)

    if col_int_type and col_int_type in int_sub.columns:
        row['n_intervention_types'] = int_sub[col_int_type].nunique()
    else:
        row['n_intervention_types'] = max(1, row['intervention_breadth'])

    int_feature_rows.append(row)

int_features = pd.DataFrame(int_feature_rows)
print(f'Intervention features built: {len(int_features)} residents')
int_features.head(3)

In [ ]:
# ── Resident baseline features ────────────────────────────────────────────────
# These are characteristics at intake — valid predictors that predate the outcome.

res_base = residents[[col_res_id]].copy() if col_res_id else residents.head(0).copy()

# Age at admission
if 'age_at_admit' in residents.columns:
    res_base = res_base.merge(residents[[col_res_id, 'age_at_admit']], on=col_res_id, how='left')

# Length of stay in care (days since admission)
if 'admit_date' in residents.columns:
    residents['days_in_care'] = (REF_DATE - residents['admit_date']).dt.days.clip(lower=0)
    res_base = res_base.merge(residents[[col_res_id, 'days_in_care']], on=col_res_id, how='left')

# Case category dummies
if col_res_category and col_res_category in residents.columns:
    cat_dummies = pd.get_dummies(
        residents[col_res_category].astype(str).str.lower().str.strip(),
        prefix='case', drop_first=False, dtype=int
    )
    res_base = pd.concat([res_base.reset_index(drop=True),
                          cat_dummies.reset_index(drop=True)], axis=1)
    case_cat_cols = list(cat_dummies.columns)
else:
    case_cat_cols = []

print(f'Baseline features: age_at_admit, days_in_care, {case_cat_cols}')

In [ ]:
# ── Assemble final modeling dataset ──────────────────────────────────────────
# Join: resident baseline + intervention features + outcome signals

model_df = (
    outcomes
    .merge(int_features,  on=col_res_id, how='inner')
    .merge(res_base,      on=col_res_id, how='left')
)

# Require at least 2 process recordings for reliable emotional signal
model_df = model_df[model_df['n_recordings'] >= 2].reset_index(drop=True)

print(f'Residents with >= 2 recordings: {len(model_df)}')
print(f'Improvement rate:               {model_df["improved"].mean():.2%}')
model_df.describe().T.round(2).head(20)

In [ ]:
# ── Exploratory Data Analysis ─────────────────────────────────────────────────
fig, axes = plt.subplots(2, 3, figsize=(16, 9))

# 1. Improvement label distribution
model_df['improved'].value_counts().plot(
    kind='bar', ax=axes[0, 0], color=['tomato', 'steelblue'], edgecolor='white'
)
axes[0, 0].set_title('Improvement Label Distribution')
axes[0, 0].set_xticklabels(['No Progress (0)', 'Improved (1)'], rotation=0)

# 2. Improvement rate by intervention breadth
breadth_imp = model_df.groupby('intervention_breadth')['improved'].mean()
breadth_imp.plot(kind='bar', ax=axes[0, 1], color='mediumseagreen', edgecolor='white')
axes[0, 1].set_title('Improvement Rate by Intervention Breadth')
axes[0, 1].set_xlabel('# Intervention Types')

# 3. Emotion slope by label
axes[0, 2].boxplot(
    [model_df[model_df['improved'] == 0]['emotion_slope'].clip(-3, 3),
     model_df[model_df['improved'] == 1]['emotion_slope'].clip(-3, 3)]
)
axes[0, 2].set_xticklabels(['No Progress', 'Improved'])
axes[0, 2].set_title('Emotion Slope by Label')

# 4. Intervention type flags vs improvement rate
int_flags = ['has_counseling', 'has_group_therapy', 'has_medical',
             'has_educational_support', 'has_vocational', 'has_family_counseling']
int_flags = [c for c in int_flags if c in model_df.columns]
flag_rates = {
    col.replace('has_', ''): model_df.groupby(col)['improved'].mean().get(1, 0)
    for col in int_flags
}
pd.Series(flag_rates).sort_values(ascending=True).plot(
    kind='barh', ax=axes[1, 0], color='darkorange', edgecolor='white'
)
axes[1, 0].set_title('Improvement Rate When Intervention Present')
axes[1, 0].set_xlabel('Fraction Improved')

# 5. Days in care vs emotion slope
if 'days_in_care' in model_df.columns:
    axes[1, 1].scatter(model_df['days_in_care'].clip(upper=1000),
                       model_df['emotion_slope'].clip(-3, 3),
                       c=model_df['improved'], cmap='coolwarm', alpha=0.6, s=40)
    axes[1, 1].set_title('Days in Care vs Emotion Slope')
    axes[1, 1].set_xlabel('Days in Care')
    axes[1, 1].set_ylabel('Emotion Slope')

# 6. N recordings vs improvement
model_df.boxplot(column='n_recordings', by='improved', ax=axes[1, 2])
axes[1, 2].set_title('Session Count by Label')
axes[1, 2].set_xlabel('improved')
plt.sca(axes[1, 2])
plt.suptitle('')

plt.suptitle('Intervention Effectiveness EDA — BrightHut', fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# ── Correlation heatmap ───────────────────────────────────────────────────────
corr_cols = [
    'improved', 'emotion_slope', 'health_slope', 'edu_attend_slope',
    'n_recordings', 'n_interventions', 'intervention_breadth',
    'avg_int_duration_days', 'pct_completed_plans', 'avg_freq',
    'has_counseling', 'has_group_therapy', 'has_medical',
    'has_educational_support', 'has_vocational', 'has_family_counseling',
    'n_individual_sessions', 'n_group_sessions',
    'n_incidents_total', 'edu_enrolled'
]
corr_cols = [c for c in corr_cols if c in model_df.columns]
corr = model_df[corr_cols].corr()

plt.figure(figsize=(13, 11))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='coolwarm',
            center=0, linewidths=0.4, annot_kws={'size': 7})
plt.title('Feature Correlation Matrix — Intervention Effectiveness')
plt.tight_layout()
plt.show()

---
## 3. Modeling & Feature Selection

In [ ]:
# ── Assemble feature matrix ───────────────────────────────────────────────────
#
# TAUTOLOGICAL EXCLUSIONS from the explanatory model:
#   emotion_slope, health_slope, edu_attend_slope, edu_perf_slope
#     → These ARE the outcome signals that define the label. Including them in the
#       explanatory model would trivially predict improvement from improvement itself.
#   emotion_improved, health_improved, education_improved, domains_improved
#     → Component indicators of the composite label.
#   last_emotion, last_health_score
#     → Post-intervention endpoint values; definitionally linked to improvement label.
#   incident_reduction
#     → Defined as improvement in incident count; partially overlaps with the label.
#
# VALID BASELINE predictors retained in the explanatory model:
#   first_emotion, first_health_score  → pre-intervention baseline state
#   n_recordings, n_individual_sessions, n_group_sessions  → dosage of counseling
#   intervention plan attributes  → modifiable program design variables
#   resident demographics  → fixed characteristics at intake

TAUTOLOGICAL = {
    'emotion_slope', 'health_slope', 'edu_attend_slope', 'edu_perf_slope',
    'emotion_improved', 'health_improved', 'education_improved', 'domains_improved',
    'last_emotion', 'last_health_score', 'incident_reduction',
    'improved'  # the label itself
}

INTERVENTION_FEATURES = [
    # Dosage
    'n_interventions', 'n_intervention_types', 'intervention_breadth',
    'avg_int_duration_days', 'total_int_duration_days', 'pct_completed_plans', 'avg_freq',
    # Type flags
    'has_counseling', 'has_group_therapy', 'has_medical',
    'has_educational_support', 'has_vocational', 'has_family_counseling',
    'has_crisis_intervention',
    # Counseling session counts (dose proxy)
    'n_recordings', 'n_individual_sessions', 'n_group_sessions',
]

BASELINE_FEATURES = [
    # Pre-intervention resident state
    'first_emotion', 'first_health_score', 'avg_emotion',
    'n_health_records', 'n_edu_records', 'edu_enrolled',
    'n_incidents_total',
    # Demographics
    'age_at_admit', 'days_in_care',
]

ALL_FEATURES = INTERVENTION_FEATURES + BASELINE_FEATURES + case_cat_cols
ALL_FEATURES = [c for c in ALL_FEATURES if c in model_df.columns]
EXPL_FEATURES = [c for c in ALL_FEATURES if c not in TAUTOLOGICAL]

model_df[ALL_FEATURES] = model_df[ALL_FEATURES].fillna(0)

X     = model_df[ALL_FEATURES].copy()
X_exp = model_df[EXPL_FEATURES].copy()
y     = model_df['improved'].copy()

print(f'Total features (predictive):      {len(ALL_FEATURES)}')
print(f'Explanatory features (no tautol): {len(EXPL_FEATURES)}')
print(f'Class balance: {y.mean():.2%} improved  ({y.sum()} / {len(y)})')

In [ ]:
# ── Stratified train / test split ────────────────────────────────────────────

if len(model_df) < 10 or y.nunique() < 2:
    X_train = X_test = X
    Xe_train = Xe_test = X_exp
    y_train = y_test = y
    print('⚠ Very small dataset — using full set for both training and evaluation.')
    print('  Treat all results as directional; validate with new data as it accumulates.')
else:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE
    )
    Xe_train = X_train[EXPL_FEATURES]
    Xe_test  = X_test[EXPL_FEATURES]

cv_n = min(5, max(2, int(y_train.sum())))
cv   = StratifiedKFold(n_splits=cv_n, shuffle=True, random_state=RANDOM_STATE)

print(f'Train: {len(X_train)}  |  Test: {len(X_test)}')
print(f'Train improvement rate: {y_train.mean():.2%}  |  Test: {y_test.mean():.2%}')
print(f'CV folds: {cv_n}')

if len(model_df) < 20:
    print()
    print('⚠ NOTE: Small resident sample (<20). The OLS explanatory model is the primary')
    print('  analytical output. Predictive AUC estimates will have wide uncertainty.')

In [ ]:
# ── Model A: Logistic Regression on EXPLANATORY features ─────────────────────
# Excludes all tautological features. Coefficients represent associations between
# modifiable program design choices and resident improvement.

pipe_lr = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    LogisticRegression(
                   class_weight='balanced', max_iter=1000,
                   C=0.5, random_state=RANDOM_STATE
               ))
])

if len(Xe_train) >= cv_n * 2 and y_train.nunique() > 1:
    cv_lr = cross_val_score(pipe_lr, Xe_train, y_train, cv=cv,
                            scoring='roc_auc', n_jobs=-1)
    print(f'LR  CV ROC-AUC (explanatory): {cv_lr.mean():.4f}  (±{cv_lr.std():.4f})')
pipe_lr.fit(Xe_train, y_train)

In [ ]:
# ── Model B: Random Forest on ALL features (predictive) ──────────────────────

pipe_rf = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    RandomForestClassifier(
                   n_estimators=300, max_depth=4,
                   min_samples_leaf=2, class_weight='balanced',
                   random_state=RANDOM_STATE, n_jobs=-1
               ))
])

if len(X_train) >= cv_n * 2 and y_train.nunique() > 1:
    cv_rf = cross_val_score(pipe_rf, X_train, y_train, cv=cv,
                            scoring='roc_auc', n_jobs=-1)
    print(f'RF  CV ROC-AUC (all features): {cv_rf.mean():.4f}  (±{cv_rf.std():.4f})')
pipe_rf.fit(X_train, y_train)

In [ ]:
# ── Model C: Gradient Boosted Trees (predictive — primary) ───────────────────

pipe_gbt = Pipeline([
    ('scaler', StandardScaler()),
    ('clf',    GradientBoostingClassifier(
                   n_estimators=200, learning_rate=0.05,
                   max_depth=3, subsample=0.8,
                   random_state=RANDOM_STATE
               ))
])

if len(X_train) >= cv_n * 2 and y_train.nunique() > 1:
    cv_gbt = cross_val_score(pipe_gbt, X_train, y_train, cv=cv,
                             scoring='roc_auc', n_jobs=-1)
    print(f'GBT CV ROC-AUC (all features): {cv_gbt.mean():.4f}  (±{cv_gbt.std():.4f})')
pipe_gbt.fit(X_train, y_train)

---
## 4. Evaluation & Selection

In [ ]:
# ── Hold-out test performance ─────────────────────────────────────────────────
results = {}
for name, pipe, X_ev in [
    ('Logistic Regression (expl.)', pipe_lr,  Xe_test),
    ('Random Forest (all)',         pipe_rf,  X_test),
    ('Gradient Boosting (all)',     pipe_gbt, X_test)
]:
    proba = pipe.predict_proba(X_ev)[:, 1]
    auc   = roc_auc_score(y_test, proba) if y_test.nunique() > 1 else float('nan')
    results[name] = {'ROC-AUC': round(auc, 4), 'proba': proba}
    print(f'{name:35s}  Test AUC = {auc:.4f}')

pred_results   = {k: v for k, v in results.items() if 'expl.' not in k}
best_pred_name = max(pred_results, key=lambda k: pred_results[k]['ROC-AUC'])
best_pred_pipe = {'Random Forest (all)':     pipe_rf,
                  'Gradient Boosting (all)': pipe_gbt}[best_pred_name]
best_proba     = pred_results[best_pred_name]['proba']
print(f'\nBest predictive model: {best_pred_name}')

In [ ]:
# ── Threshold tuning with F2 (recall-weighted) ────────────────────────────────
# Missing a resident predicted to improve who actually does not is low-cost (we
# leave them on a working plan). Missing a resident predicted NOT to improve who
# would benefit from plan revision is a care failure. We weight recall at 2× precision.

eval_X = X_test if len(X_test) > 0 else X_train
eval_y = y_test if len(y_test) > 0 else y_train

if eval_y.nunique() > 1:
    precisions, recalls, thresholds = precision_recall_curve(eval_y, best_proba)
    f2_scores = [
        fbeta_score(eval_y, (best_proba >= t).astype(int), beta=2.0)
        for t in thresholds
    ]
    best_idx       = int(np.argmax(f2_scores))
    BEST_THRESHOLD = float(thresholds[best_idx])

    print(f'Optimal threshold (max F2): {BEST_THRESHOLD:.3f}')
    print(f'  Precision: {precisions[best_idx]:.3f}')
    print(f'  Recall:    {recalls[best_idx]:.3f}')
    print(f'  F2:        {f2_scores[best_idx]:.3f}')

    plt.figure(figsize=(8, 4))
    plt.plot(thresholds, f2_scores, color='teal')
    plt.axvline(BEST_THRESHOLD, color='red', linestyle='--',
                label=f'Best threshold = {BEST_THRESHOLD:.3f}')
    plt.xlabel('Decision Threshold')
    plt.ylabel('F2 Score')
    plt.title('F2-Optimized Threshold — Improvement Classifier')
    plt.legend()
    plt.tight_layout()
    plt.show()

    y_pred = (best_proba >= BEST_THRESHOLD).astype(int)
    print('\nClassification Report:')
    print(classification_report(eval_y, y_pred,
                                target_names=['Stalled / Declining', 'Improving']))

    fig, ax = plt.subplots(figsize=(5, 4))
    ConfusionMatrixDisplay.from_predictions(
        eval_y, y_pred,
        display_labels=['Stalled/Declining', 'Improving'],
        cmap='Blues', ax=ax
    )
    ax.set_title(f'{best_pred_name}\n@ threshold={BEST_THRESHOLD:.2f}')
    plt.tight_layout()
    plt.show()
else:
    BEST_THRESHOLD = 0.5
    print('Threshold tuning skipped — evaluation set has only one class.')

In [ ]:
# ── Permutation importance on evaluation set ──────────────────────────────────

if eval_y.nunique() > 1 and len(eval_X) >= 4:
    perm = permutation_importance(
        best_pred_pipe, eval_X, eval_y,
        n_repeats=20, scoring='roc_auc',
        random_state=RANDOM_STATE, n_jobs=-1
    )
    perm_df = pd.DataFrame({
        'feature':    ALL_FEATURES,
        'importance': perm.importances_mean,
        'std':        perm.importances_std
    }).sort_values('importance', ascending=False).head(20)

    plt.figure(figsize=(9, 6))
    plt.barh(perm_df['feature'][::-1], perm_df['importance'][::-1],
             xerr=perm_df['std'][::-1], color='darkorange', alpha=0.85)
    plt.xlabel('Mean Decrease in ROC-AUC')
    plt.title(f'Feature Importances (Permutation) — {best_pred_name}')
    plt.tight_layout()
    plt.show()

    print('Top 10 features:')
    print(perm_df.head(10).to_string(index=False))
else:
    print('Permutation importance skipped — test set too small or single-class.')

---
## 5. Causal and Relationship Analysis

### 5a. OLS Regression on Continuous Improvement Score (Explanatory — Primary)

We use OLS on the continuous `domains_improved` count (0, 1, 2, or 3 domains showing progress) rather than binary classification, which gives richer coefficient estimates. This answers: *"Holding resident baseline severity constant, how much does each intervention type or dosage attribute change the expected number of domains that improve?"*

**Tautological features correctly excluded:**
The explanatory model uses only:
- **Pre-intervention baseline state** (first emotion, first health score, age, days in care)
- **Intervention plan attributes** (types, duration, frequency, breadth, completion rate)
- **Counseling dosage** (session counts by type)

It excludes all post-intervention outcome signals (emotion slope, health slope, education slope, last scores) that are definitionally linked to the label.

### Selection Bias Caveat

Residents are not randomly assigned to interventions. Social workers assign more intensive or specialized interventions to residents with more severe presentations. This means a negative raw correlation between an intervention type and improvement does **not** mean the intervention is harmful — it may simply be assigned preferentially to the hardest cases. The OLS results should be read as *"among residents with similar baseline severity, what intervention design attributes are associated with better outcomes?"* — not as clean causal estimates.

In [ ]:
# ── OLS on domains_improved (0–3 continuous outcome) ─────────────────────────

y_ols     = model_df['domains_improved'].fillna(0)
X_ols_raw = model_df[EXPL_FEATURES].fillna(0)

# Standardize for coefficient comparability
X_ols_std = (X_ols_raw - X_ols_raw.mean()) / (X_ols_raw.std().replace(0, 1))
X_ols_c   = sm.add_constant(X_ols_std)

if len(model_df) >= len(EXPL_FEATURES) + 3:
    ols_model = sm.OLS(y_ols, X_ols_c).fit()
    print(ols_model.summary())
    ols_fitted = True
else:
    print('⚠ More features than observations — switching to Ridge regression.')
    ridge = Ridge(alpha=1.0, fit_intercept=True)
    ridge.fit(X_ols_std, y_ols)
    ridge_df = pd.DataFrame({'feature': EXPL_FEATURES, 'ridge_coef': ridge.coef_})\
                 .sort_values('ridge_coef', key=abs, ascending=False)
    print('Ridge coefficients (standardized features — effect on # domains improved):')
    print(ridge_df.to_string(index=False))
    ols_model  = None
    ols_fitted = False

In [ ]:
# ── Plot OLS coefficients ─────────────────────────────────────────────────────
if ols_fitted:
    coef_df = pd.DataFrame({
        'feature': ols_model.params.index,
        'coef':    ols_model.params.values,
        'ci_lo':   ols_model.conf_int()[0].values,
        'ci_hi':   ols_model.conf_int()[1].values,
        'pvalue':  ols_model.pvalues.values
    }).query("feature != 'const'")
    coef_df['abs_coef'] = coef_df['coef'].abs()
    coef_df = coef_df.sort_values('abs_coef', ascending=False).head(20)

    colors = ['steelblue' if c > 0 else 'tomato' for c in coef_df['coef']]
    plt.figure(figsize=(10, 7))
    plt.barh(coef_df['feature'][::-1], coef_df['coef'][::-1],
             color=colors[::-1], alpha=0.85)
    plt.errorbar(
        coef_df['coef'][::-1], range(len(coef_df)),
        xerr=[
            (coef_df['coef'] - coef_df['ci_lo'])[::-1].values,
            (coef_df['ci_hi'] - coef_df['coef'])[::-1].values
        ],
        fmt='none', color='black', capsize=3, linewidth=1
    )
    plt.axvline(0, color='black', linewidth=0.8)
    plt.xlabel('Standardized OLS Coefficient (effect on # domains improved)')
    plt.title('Explanatory Model: Intervention Attribute Effects on Resident Improvement\n'
              '(Blue = associated with more domains improving, Red = fewer)')
    plt.tight_layout()
    plt.show()

    print(f'\nModel adj. R²: {ols_model.rsquared_adj:.4f}')
    print('\nStatistically significant features (p < 0.10):')
    sig = coef_df[coef_df['pvalue'] < 0.10].sort_values('coef', ascending=False)
    print(sig[['feature','coef','pvalue']].to_string(index=False))

In [ ]:
# ── 5b. Logistic regression odds ratios for binary improved label ─────────────

X_logit_c = sm.add_constant(
    (X_ols_raw - X_ols_raw.mean()) / (X_ols_raw.std().replace(0, 1))
)

if len(model_df) >= len(EXPL_FEATURES) + 5 and y.nunique() > 1:
    try:
        logit_model = sm.Logit(y, X_logit_c).fit(disp=False, maxiter=300)
        or_df = pd.DataFrame({
            'feature':    logit_model.params.index,
            'log_odds':   logit_model.params.values,
            'odds_ratio': np.exp(logit_model.params.values),
            'pvalue':     logit_model.pvalues.values
        }).query("feature != 'const'").sort_values('log_odds', ascending=False)

        print('Odds Ratios — binary improvement label (per 1 SD change in feature):')
        print(or_df[['feature','odds_ratio','pvalue']].to_string(index=False))

        # Forest plot
        or_ci = logit_model.conf_int()
        or_df['or_lo'] = np.exp(or_ci[0].values[1:])
        or_df['or_hi'] = np.exp(or_ci[1].values[1:])

        fig, ax = plt.subplots(figsize=(9, 7))
        y_pos  = range(len(or_df))
        colors = ['steelblue' if v > 1 else 'tomato' for v in or_df['odds_ratio']]
        ax.barh(list(y_pos), or_df['log_odds'].values, color=colors, alpha=0.8)
        ax.errorbar(
            or_df['log_odds'].values, list(y_pos),
            xerr=[
                (or_df['log_odds'] - np.log(or_df['or_lo'])).values,
                (np.log(or_df['or_hi']) - or_df['log_odds']).values
            ],
            fmt='none', color='black', capsize=3, linewidth=1
        )
        ax.set_yticks(list(y_pos))
        ax.set_yticklabels(or_df['feature'].tolist())
        ax.axvline(0, color='black', linewidth=0.8)
        ax.set_xlabel('Log Odds Ratio (per 1 SD — higher = more likely to improve)')
        ax.set_title('Intervention Feature Associations with Resident Improvement (Odds Ratios)')
        plt.tight_layout()
        plt.show()
    except Exception as e:
        print(f'Logistic regression failed (likely singular matrix at this sample size): {e}')
        print('This is expected with fewer than ~30 residents; rely on OLS + Ridge above.')
else:
    print('Logistic regression skipped — insufficient observations.')

In [ ]:
# ── 5c. Intervention type × improvement rate breakdown ────────────────────────
# Actionable: what fraction of residents receiving each intervention type improved?

print('=== Improvement rate by intervention type (raw, unadjusted for severity) ===')
print(f'  ⚠ Selection bias note: more intensive interventions may go to harder cases.')
print()

flag_cols = ['has_counseling', 'has_group_therapy', 'has_medical',
             'has_educational_support', 'has_vocational', 'has_family_counseling',
             'has_crisis_intervention']
flag_cols = [c for c in flag_cols if c in model_df.columns]

print(f"{'Intervention Type':30s}  {'N With':>7s}  {'Impr Rate (With)':>17s}  "
      f"{'N Without':>10s}  {'Impr Rate (W/O)':>16s}")
print('-' * 90)
for col in flag_cols:
    with_df    = model_df[model_df[col] == 1]
    without_df = model_df[model_df[col] == 0]
    r_with     = with_df['improved'].mean()    if len(with_df)    > 0 else float('nan')
    r_without  = without_df['improved'].mean() if len(without_df) > 0 else float('nan')
    label      = col.replace('has_', '')
    print(f'  {label:28s}  {len(with_df):>7d}  {r_with:>17.1%}  '
          f'{len(without_df):>10d}  {r_without:>16.1%}')

# Dosage analysis: does more sessions always help?
print('\n=== Improvement rate by counseling session count quartile ===')
model_df['sessions_quartile'] = pd.qcut(
    model_df['n_recordings'], q=4,
    labels=['Q1 (few)','Q2','Q3','Q4 (many)'], duplicates='drop'
)
print(model_df.groupby('sessions_quartile')['improved'].agg(['mean','count']).round(3))

# Breadth: does more types of intervention help?
print('\n=== Improvement rate by intervention breadth ===')
print(model_df.groupby('intervention_breadth')['improved'].agg(['mean','count']).round(3))

---
## 6. Deployment Notes

### What to Deploy

Two artifacts are produced:

1. **Improvement classifier** (`intervention_effectiveness_model.pkl`) — The best-performing GBT/RF pipeline, used to score active residents' current improvement trajectory and flag those predicted to stall.
2. **Effectiveness config** (`intervention_effectiveness_config.json`) — Operating threshold, feature lists, OLS coefficients, and model metadata.

### Integration Points

| Deployment Target | Description |
|---|---|
| `GET /api/ml/intervention-effectiveness` | Admin/staff endpoint; returns per-resident improvement scores and stall flags |
| Caseload Inventory | Improvement trajectory badge on each resident row (Improving / On Track / Review Needed) |
| Process Recording Form | After each new recording entry, recalculate the resident's predicted trajectory and surface it to the social worker |
| Case Conference View | Pre-meeting summary: "Predicted improvement trajectory: STALLED — consider plan revision" |
| Admin Dashboard | Safehouse-level aggregate: % residents currently on improving trajectory |
| Donor-Facing Impact Dashboard | Anonymized: "X% of residents in care are showing measurable improvements across health, education, and emotional wellbeing" — closes the loop between donations and outcomes |

### Retraining Schedule

- **Monthly retraining** recommended as new process recordings, health records, and education records accumulate
- Retrain trigger: AUC drops below 0.58 on a 60-day rolling holdout of newly admitted residents
- Drift signal: monitor distribution of `first_emotion` scores at intake — if case severity changes (e.g., org takes in more crisis cases), baseline features shift

### Business Recommendations from Explanatory Model

Expected findings from the OLS and odds ratio analyses, based on rehabilitation research and the data design:

- **Breadth over intensity:** Residents receiving a wider variety of intervention types (higher `intervention_breadth`) are expected to show more domains improving than residents receiving very intensive interventions in a single domain. Cross-domain programming appears to produce more resilient improvement.
- **Plan completion matters:** Higher `pct_completed_plans` is expected to have a positive coefficient — residents whose plans are fully delivered (not abandoned mid-course) show better outcomes, which may reflect both plan quality and resident engagement.
- **Educational engagement accelerates emotional recovery:** The `has_educational_support` and `edu_enrolled` features are expected to be positively associated with emotional improvement as well as the education domain — structured activity provides routine and purpose that reinforces counseling gains.
- **Family counseling is double-edged:** `has_family_counseling` may show mixed results depending on family cooperation levels. For residents from cooperative families, family involvement strongly predicts improvement. For those from non-cooperative or unsafe family environments, it may not.
- **Baseline severity is the strongest predictor of trajectory:** The `first_emotion` and `first_health_score` baseline features are expected to have high importance — residents who enter with higher baseline functioning have more capacity for measurable gains. This is not a reason to withhold services from severely presenting residents; it is a reason to set appropriate expectations and interpret improvement rates in context.

⚠️ **Causality caveat:** All findings from this observational study are subject to selection bias. The OLS model controls for observed baseline severity, but unobserved confounders (family support, community resources, individual resilience) affect both intervention assignment and outcomes. These results should inform program design decisions as working hypotheses, not as definitive evidence of effectiveness.

In [ ]:
# ── Save model artifacts ──────────────────────────────────────────────────────
ARTIFACT_DIR = Path('model_artifacts')
ARTIFACT_DIR.mkdir(exist_ok=True)

model_path  = ARTIFACT_DIR / 'intervention_effectiveness_model.pkl'
config_path = ARTIFACT_DIR / 'intervention_effectiveness_config.json'

joblib.dump(best_pred_pipe, model_path)
print(f'Model saved → {model_path}')

ols_coefs = {}
if ols_fitted:
    ols_coefs = {
        feat: {'coef': round(float(c), 6), 'pvalue': round(float(p), 4)}
        for feat, c, p in zip(
            ols_model.params.index, ols_model.params.values, ols_model.pvalues.values
        ) if feat != 'const'
    }

config = {
    'model_type':            best_pred_name,
    'threshold':             round(BEST_THRESHOLD, 4),
    'all_feature_cols':      ALL_FEATURES,
    'expl_feature_cols':     EXPL_FEATURES,
    'tautological_excluded': list(TAUTOLOGICAL),
    'ols_coefficients':      ols_coefs,
    'ols_adj_r2':            round(float(ols_model.rsquared_adj), 4) if ols_fitted else None,
    'target':                'improved',
    'improvement_definition': 'positive direction in >= 2 of 3 domains (emotion, health, education)',
    'min_recordings_required': 2,
    'notes':                 ('Retrain monthly as new process recordings accumulate. '
                              'Interpret OLS coefficients in context of selection bias — '
                              'residents are not randomly assigned to interventions.'),
    'api_endpoint':          'GET /api/ml/intervention-effectiveness',
    'auth_required':         'admin or staff'
}

with open(config_path, 'w') as f:
    json.dump(config, f, indent=2)
print(f'Config saved → {config_path}')

In [ ]:
# ── Inference function for API integration ────────────────────────────────────

def score_residents_improvement() -> pd.DataFrame:
    """
    Score all active residents on their current improvement trajectory.

    Returns a DataFrame with one row per resident, sorted by improvement
    probability (ascending — residents most at risk of stalling appear first).

    Columns:
        - resident_id
        - improvement_score   : float  probability of composite improvement (0–1)
        - predicted_improving : int    1 = improving, 0 = stalled/declining
        - status_label        : str    'IMPROVING' / 'ON TRACK' / 'REVIEW NEEDED'
        - n_recordings        : int    number of counseling sessions on record
        - domains_improved    : int    domains currently showing positive direction
    """
    with open(config_path) as f:
        cfg = json.load(f)

    loaded_model = joblib.load(model_path)

    X_inf = model_df[cfg['all_feature_cols']].fillna(0)
    scores = loaded_model.predict_proba(X_inf)[:, 1]

    output = model_df[[col_res_id, 'n_recordings', 'domains_improved']].copy()
    output['improvement_score']   = scores.round(4)
    output['predicted_improving'] = (scores >= cfg['threshold']).astype(int)
    output['status_label'] = pd.cut(
        output['improvement_score'],
        bins=[0, 0.35, 0.65, 1.01],
        labels=['REVIEW NEEDED', 'ON TRACK', 'IMPROVING'],
        include_lowest=True
    )

    # Residents below the recording threshold get a separate flag
    insufficient = outcomes[outcomes['n_recordings'] < cfg['min_recordings_required']][[col_res_id]].copy()
    if len(insufficient) > 0:
        insufficient['improvement_score']   = None
        insufficient['predicted_improving'] = None
        insufficient['status_label']        = 'INSUFFICIENT DATA'
        insufficient['n_recordings']        = outcomes.loc[
            outcomes[col_res_id].isin(insufficient[col_res_id]), 'n_recordings'
        ].values
        insufficient['domains_improved']    = None
        output = pd.concat([output, insufficient], ignore_index=True)

    # Sort: REVIEW NEEDED first so staff see them at top of caseload view
    status_order = {'REVIEW NEEDED': 0, 'ON TRACK': 1, 'IMPROVING': 2, 'INSUFFICIENT DATA': 3}
    output['sort_key'] = output['status_label'].map(status_order).fillna(4)
    output = output.sort_values(['sort_key', 'improvement_score']).drop(columns='sort_key')

    return output.reset_index(drop=True)


# ── Run and preview ───────────────────────────────────────────────────────────
resident_scores = score_residents_improvement()
print(f'Scored {len(resident_scores)} residents')
print('\nStatus breakdown:')
print(resident_scores['status_label'].value_counts())
print('\nResidents flagged for review (top priority):')
print(resident_scores[resident_scores['status_label'] == 'REVIEW NEEDED'].to_string(index=False))